# Ruhrwatt: explore the silver tables

Reads `data/silver/energy_15min.parquet` and `energy_daily.parquet` (build them with `make silver`).
Values are MWh per quarter-hour; `amp_` = Amprion control area (NRW proxy), `de_` = Germany.
Data: Bundesnetzagentur | SMARD.de, CC BY 4.0.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

# RUHRWATT_DATA_DIR if set, else <project root>/data (works from the repo or notebooks/).
if "RUHRWATT_DATA_DIR" in os.environ:
    DATA = Path(os.environ["RUHRWATT_DATA_DIR"]).expanduser()
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    DATA = ROOT / "data"
SILVER = DATA / "silver"

qh = pl.read_parquet(SILVER / "energy_15min.parquet")
daily = pl.read_parquet(SILVER / "energy_daily.parquet").filter(pl.col("is_complete_day"))
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
print(f"15-min rows: {qh.height:,}  complete days: {daily.height}")
print(f"from {daily['date_berlin'].min()} to {daily['date_berlin'].max()}")

## Daily base and peak price

In [ ]:
fig, ax = plt.subplots()
ax.plot(daily["date_berlin"], daily["price_base"], lw=1, label="base")
ax.plot(daily["date_berlin"], daily["price_peak"], lw=1, alpha=0.7, label="peak (Mon-Fri 8-20)")
ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("EUR/MWh")
ax.set_title("Day-ahead price DE-LU, daily averages")
ax.legend();

## Negative-price hours per month

In [ ]:
monthly = (
    daily.group_by(pl.col("date_berlin").dt.truncate("1mo").alias("month"))
    .agg(
        (pl.col("price_negative_qh").sum() / 4).alias("negative_hours"),
        pl.col("price_base").mean().alias("price_base"),
        (pl.col("de_wind_pv_mwh").sum() / pl.col("de_load_mwh").sum()).alias("de_wind_pv_share"),
        (pl.col("amp_wind_pv_mwh").sum() / pl.col("amp_load_mwh").sum()).alias("amp_wind_pv_share"),
    )
    .sort("month")
)
fig, ax = plt.subplots()
ax.bar(monthly["month"], monthly["negative_hours"], width=20)
ax.set_ylabel("hours")
ax.set_title("Hours with a negative day-ahead price")
monthly

## Wind + PV share of load: Germany vs Amprion (NRW)

In [ ]:
fig, ax = plt.subplots()
ax.plot(monthly["month"], monthly["de_wind_pv_share"] * 100, marker="o", label="Germany")
ax.plot(monthly["month"], monthly["amp_wind_pv_share"] * 100, marker="o", label="Amprion")
ax.set_ylabel("% of load")
ax.set_title("Onshore wind + PV as share of load (monthly)")
ax.legend();

## Average price by hour of day

Where an NRW plant with flexible load could shift consumption: workdays vs weekends,
native 15-minute prices only (from 2025-10-01).

In [ ]:
profile = (
    qh.filter(pl.col("price_is_native_15min") & pl.col("price_eur_mwh").is_not_null())
    .group_by("hour_berlin", "is_workday")
    .agg(pl.col("price_eur_mwh").mean())
    .sort("hour_berlin")
)
fig, ax = plt.subplots()
for workday, label in [(True, "workday"), (False, "weekend / holiday")]:
    d = profile.filter(pl.col("is_workday") == workday)
    ax.plot(d["hour_berlin"], d["price_eur_mwh"], marker="o", label=label)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour (Berlin)")
ax.set_ylabel("EUR/MWh")
ax.set_title("Average price by hour of day")
ax.legend();

## Price vs residual load (Germany)

In [ ]:
s = qh.filter(pl.col("is_complete")).sample(min(20_000, qh.height), seed=0)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(s["de_residual_load_mwh"] * 4 / 1000, s["price_eur_mwh"], s=2, alpha=0.2)
ax.set_xlabel("residual load, GW")
ax.set_ylabel("EUR/MWh")
ax.set_title("Price rises with residual load (load minus wind and PV)");